# 13_qdrant: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/13_qdrant/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Build typed payload postings for two fictional sellers.

Run: uv run python modules/13_qdrant/examples/01_from_scratch.py
Concepts: Build typed payload postings for two fictional sellers.
Expected: IDs matching both postings.
CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
"""

from common.data import records_from_products
from common.stores.filters import parse_filter
from common.teaching import catalog
from tinyvec.filters import MetadataIndex


def main() -> None:
    # KEY IDEA: Build typed payload postings for two fictional sellers.
    rows, matrix, _, _ = catalog(160)
    records = records_from_products(rows, matrix)
    expr = parse_filter(
        {
            "op": "and",
            "clauses": [
                {"op": "eq", "field": "region", "value": "north"},
                {"op": "lt", "field": "price", "value": 35},
            ],
        }
    )
    eligible = MetadataIndex(records).eligible(expr)
    assert eligible == {
        i
        for i, r in enumerate(records)
        if r.metadata["region"] == "north" and r.metadata["price"] < 35
    }
    print("eligible IDs", [records[i].id for i in sorted(eligible)])


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/13_qdrant/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None or "qdrant" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra qdrant; start qdrant and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Configure a Qdrant payload index and quantized dense search.

    Run: uv run --extra qdrant python modules/13_qdrant/examples/02_production.py
    Concepts: Configure a Qdrant payload index and quantized dense search.
    Expected: native payload index and quantized search.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the qdrant profile first.
    """

    # course-extra: qdrant
    # course-profile: qdrant
    from qdrant_client import models as m

    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Configure a Qdrant payload index and quantized dense search.
        with session("qdrant", quantize=True) as (store, _, q):
            store.db.create_payload_index(
                store.name, "region", field_schema=m.PayloadSchemaType.KEYWORD, wait=True
            )
            filt = m.Filter(
                must=[m.FieldCondition(key="region", match=m.MatchValue(value="north"))]
            )
            points = store.db.query_points(
                store.name,
                query=q.tolist(),
                using="dense",
                query_filter=filt,
                limit=5,
                with_payload=True,
                search_params=m.SearchParams(
                    quantization=m.QuantizationSearchParams(rescore=True, oversampling=2)
                ),
            ).points
            assert points and all(p.payload["region"] == "north" for p in points)
            print(
                "north region",
                [p.id for p in points],
                store.db.get_collection(store.name).payload_schema,
            )

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "qdrant")

## 03_measure

Source: `modules/13_qdrant/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None or "qdrant" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra qdrant; start qdrant and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Measure a controlled database comparison on the same catalog.

    Run: uv run --extra qdrant python modules/13_qdrant/examples/03_measure.py
    Concepts: Measure a controlled database comparison on the same catalog.
    Expected: filtered and keyword query sweep.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the qdrant profile first.
    """

    # course-extra: qdrant
    # course-profile: qdrant
    from common.tier3 import measure

    def main() -> None:
        # KEY IDEA: Measure a controlled database comparison on the same catalog.
        for quantized in [False, True]:
            measure(
                "qdrant", parameters={"quantize": quantized}, query_types=["filtered", "keyword"]
            )

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "qdrant")

## 04_hybrid_multivector

Source: `modules/13_qdrant/examples/04_hybrid_multivector.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None or "qdrant" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra qdrant; start qdrant and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Use dense, sparse and multiple vectors through the Qdrant Query API.

    Run: uv run --extra qdrant python modules/13_qdrant/examples/04_hybrid_multivector.py
    Concepts: Use dense, sparse and multiple vectors through the Qdrant Query API.
    Expected: named-vector fusion and MaxSim.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the qdrant profile first.
    """

    # course-extra: qdrant
    # course-profile: qdrant
    import uuid
    from contextlib import closing

    from qdrant_client import QdrantClient
    from qdrant_client import models as m

    from common.settings import setting
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Use dense, sparse and multiple vectors through the Qdrant Query API.
        with session("qdrant") as (store, rows, q):
            hits = store.hybrid_search(rows.iloc[0].sku, q, 5)
            print("fused named fields", [h.id for h in hits])
        with closing(QdrantClient(url=setting("QDRANT_URL", "http://127.0.0.1:6333"))) as db:
            name = "marlowe_" + uuid.uuid4().hex
            db.create_collection(
                name,
                vectors_config={
                    "tokens": m.VectorParams(
                        size=2,
                        distance=m.Distance.COSINE,
                        multivector_config=m.MultiVectorConfig(
                            comparator=m.MultiVectorComparator.MAX_SIM
                        ),
                    )
                },
            )
            try:
                db.upsert(
                    name,
                    points=[
                        m.PointStruct(id=1, vector={"tokens": [[1.0, 0.0], [0.0, 1.0]]}),
                        m.PointStruct(id=2, vector={"tokens": [[1.0, 0.0]]}),
                    ],
                    wait=True,
                )
                points = db.query_points(
                    name, query=[[1.0, 0.0], [0.0, 1.0]], using="tokens"
                ).points
                assert points[0].id == 1
                print("two-token coverage", [(p.id, p.score) for p in points])
            finally:
                db.delete_collection(name)

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "qdrant")

## 05_snapshots_tenants

Source: `modules/13_qdrant/examples/05_snapshots_tenants.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None or "qdrant" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra qdrant; start qdrant and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Restore a real Qdrant snapshot to a new isolated collection.

    Run: uv run --extra qdrant python modules/13_qdrant/examples/05_snapshots_tenants.py
    Concepts: Restore a real Qdrant snapshot to a new isolated collection.
    Expected: restored native snapshot IDs.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the qdrant profile first.
    """

    # course-extra: qdrant
    # course-profile: qdrant
    import uuid

    import httpx

    from common.settings import setting
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Restore a real Qdrant snapshot to a new isolated collection.
        with (
            session("qdrant", 80) as (store, _, q),
            httpx.Client(
                base_url=setting("QDRANT_URL", "http://127.0.0.1:6333"), timeout=60
            ) as http,
        ):
            expected = store._query(q, None, 3)
            snap = store.db.create_snapshot(store.name)
            target = "marlowe_restore_" + uuid.uuid4().hex
            try:
                blob = (
                    http.get(f"/collections/{store.name}/snapshots/{snap.name}")
                    .raise_for_status()
                    .content
                )
                response = http.post(
                    f"/collections/{target}/snapshots/upload",
                    params={"priority": "snapshot", "wait": "true"},
                    files={"snapshot": (snap.name, blob, "application/octet-stream")},
                )
                response.raise_for_status()
                pts = store.db.query_points(target, query=q.tolist(), using="dense", limit=3).points
                assert [p.id for p in pts] == [key for key, _ in expected]
                print("snapshot bytes", len(blob), "restored", len(pts))
            finally:
                if store.db.collection_exists(target):
                    store.db.delete_collection(target)
                store.db.delete_snapshot(store.name, snap.name)

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "qdrant")

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.teaching import catalog

rows, _, _, _ = catalog(100)
print(rows.groupby("region").price.median())
# Change the group or price threshold, then repeat the native comparison for 13_qdrant.

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.